In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone # --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


# 1. knowledge base 구성을 위한 데이터 생성

In [8]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 3.8 s
Wall time: 3.78 s


180

In [ ]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [1]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [15]:
len(embedding.embed_query("샘플"))

4096

In [16]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [2]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# 데이터를 처음 저장할 때
index_name = 'tax-index'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 344 ms
Wall time: 1.71 s


In [6]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


# 2. 답변 생성을 위한 Retrieval

In [3]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [10]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n','\n') for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [13]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [12]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [14]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [15]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉이 5천만 원인 직장인의 경우, 소득세액을 계산하기 위해서는 먼저 종합소득세의 기본세율, 공제, 그리고 세액공제를 고려해야 합니다. 아래는 일반적인 계산 절차입니다.

1. 총급여액 = 50,000,000원
2. 근로소득공제 = 근로소득공제 표에 따라 계산
   - 총급여 5천만원인 경우 근로소득공제는 약 760만원(구체적인 산출 공식에 따라 달라질 수 있으나, 참고로 위 표에서 7천만원 이하 구간 공제액은 점차 증가)
3. 과세표준 산출
   - 과세표준 = 총급여액 - 근로소득공제 - 기타 공제(인적공제, 보험료공제 등)
   - 여기서는 인적공제(본인 150만원, 부양가족 등), 보험료공제, 의료비 공제 등을 고려하지 않고, 단순 공제만 고려시 계산
4. 소득세율 적용
   - 5천만원의 과세표준에 대한 세율은 아래와 같음.
   
   | 과세표준 구간 | 세율 | 누진공제 |
   |--------------|--------|----------|
   | 1,200만원 이하 | 6% | 0원 |
   | 1,200만원 초과 ~ 4,600만원 이하 | 15% | 108만원 |
   | 4,600만원 초과 ~ 8,800만원 이하 | 24% | 522만원 |
   | 8,800만원 초과 ~ 1억 5천만원 이하 | 35% | 1,490만원 |
   | 1억 5천만원 초과 ~ 3억 이하 | 38% | 2,595만원 |
   
   ※ 참고로, 과세표준이 대략 4,600만원에서 5천만원인 경우 세율 24%를 적용하는데, 이때 세액은  
   (과세표준 - 세율구간 시작점) × 세율 + 누진공제액으로 계산됩니다.
   
5. 세액 계산 예시
   - 대략 과세표준을 4,400만~4,700만원 구간으로 가정하고,
   - 과세표준이 5천만원인 경우, 세율 24% 적용 →  
     세액 = (5,000만원 × 0.24) - 522만원 = 약 678만원 (세금액은 공제, 세율, 기타 공제에 따라 달라질 수 있음)
   
6. 인적공제, 보험료공제, 특별세액공제 